In [1]:
!nvidia-smi

Tue Oct 20 20:04:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%mkdir -p /content/work
%cd /content/work
!git clone -b repro https://github.com/Siddharth-magesh/sam-replication.git sam
!ls sam

/content/work
Cloning into 'sam'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 98 (delta 1), reused 1 (delta 1), pack-reused 92 (from 1)
Receiving objects: 100% (98/98), 303.28 KiB | 13.79 MiB/s, done.
Resolving deltas: 100% (36/36), done.
autoaugment   figures	   LICENSE   OWNERS	requirements.txt
CONTRIBUTING  __init__.py  notes.md  README.md	sam_jax


In [3]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [4]:
!mamba create -n sam python=3.8 -y
!mamba install -n sam -c conda-forge cudatoolkit=11.0 cudnn -y

[+] 0.0s
[+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64  13%
conda-forge/noarch    ⣾  [+] 0.3s
conda-forge/linux-64  27%
conda-forge/noarch     1%[+] 0.4s
conda-forge/linux-64  34%
conda-forge/noarch    13%[+] 0.5s
conda-forge/linux-64  39%
conda-forge/noarch    24%[+] 0.6s
conda-forge/linux-64  41%
conda-forge/noarch    28%[+] 0.7s
conda-forge/linux-64  47%
conda-forge/noarch    40%[+] 0.8s
conda-forge/linux-64  52%
conda-forge/noarch    49%[+] 0.9s
conda-forge/linux-64  57%
conda-forge/noarch    60%[+] 1.0s
conda-forge/linux-64  60%
conda-forge/noarch    65%[+] 1.1s
conda-forge/linux-64  64%
conda-forge/noarch    75%[+] 1.2s
conda-forge/linux-64  65%
conda-forge/noarch    78%[+] 1.3s
conda-forge/linux-64  69%
conda-forge/noarch    84%[+] 1.4s
conda-forge/linux-64  72%
conda-forge/noarch    91%[+] 1.5s
conda-forge/linux-64  75%
conda-forge/noarch    98%conda-forge/noarch                                
[+] 1.6s
conda-forge/linux-64  79%[+] 1.

In [5]:
PY = "/usr/local/envs/sam/bin/python"

# drop jaxlib, relax the absl-py pin
!sed -e '/^jaxlib/d' -e 's/^absl-py==0.11.0/absl-py==0.10.0/' /content/work/sam/requirements.txt > /tmp/req_fixed.txt
!grep -E '^(absl|jax)' /tmp/req_fixed.txt      # should show absl-py==0.10.0 and jax==0.2.6, no jaxlib

# GPU jaxlib FIRST, from the CUDA-specific index
!{PY} -m pip install "jaxlib==0.1.57+cuda110" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

absl-py==0.10.0
jax==0.2.6
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 56.0 MB/s eta 0:00:00


In [6]:
!{PY} -m pip install -r /tmp/req_fixed.txt
!{PY} -m pip list 2>/dev/null | grep -i -E "^(jax|jaxlib|flax|numpy|tensorflow|absl)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.2/829.2 kB 25.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25

In [2]:
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/sam/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

!/usr/local/envs/sam/bin/python -c "import jax, jax.numpy as jnp; print(jax.devices()); x=jnp.ones((4096,4096)); print((x@x).block_until_ready().sum())"
!/usr/local/envs/sam/bin/python -c "import tensorflow as tf, tensorflow_datasets as tfds, flax; print(tf.__version__, flax.__version__)"

[GpuDevice(id=0)]
2026-10-20 20:09:21.807679: W external/org_tensorflow/tensorflow/compiler/xla/service/gpu/nvptx_compiler.cc:70] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
2026-10-20 20:09:21.807732: W external/org_tensorflow/tensorflow/compiler/xla/service/gpu/nvptx_compiler.cc:71] Searched for CUDA in the following directories:
2026-10-20 20:09:21.807751: W external/org_tensorflow/tensorflow/compiler/xla/service/gpu/nvptx_compiler.cc:74]   ./cuda_sdk_lib
2026-10-20 20:09:21.807764: W external/org_tensorflow/tensorflow/compiler/xla/service/gpu/nvptx_compiler.cc:74]   /usr/local/cuda-11.0
2026-10-20 20:09:21.807777: W external/org_tensorflow/tensorflow/compiler/xla/service/gpu/nvptx_compiler.cc:74]   .
2026-10-20 20:09:21.807789: W external/org_tensorflow/tensorflow/compiler/xla/service/gpu/nvptx_compiler.cc:76] You can choose the search directory by setting x

In [3]:
%cd /content/work
!mkdir -p /content/runs
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 \
  --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sam \
  --num_epochs 2 \
  --batch_size 256 \
  --learning_rate 0.1 \
  --weight_decay 0.0005 \
  --sam_rho 0.05 \
  --run_seed 0 2>&1 | tee /content/runs/smoke_sam.log

/content/work
2026-10-20 20:10:31.444032: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/lib:/usr/lib64-nvidia
2026-10-20 20:10:33.238696: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/lib:/usr/lib64-nvidia
2026-10-20 20:10:33.239009: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcublas.so.10'; dlerror: libcublas.so.10: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/lib:/usr/lib64-nvidia
202

In [5]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tfds_cache
!cp -r /root/tensorflow_datasets/* /content/drive/MyDrive/tfds_cache/
!ls /content/drive/MyDrive/tfds_cache

Mounted at /content/drive
cifar10  downloads


In [6]:
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/sam/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
%cd /content/work

/content/work


In [7]:
import glob, os
cands = (glob.glob("/usr/local/envs/sam/**/libdevice*.bc", recursive=True)
         or glob.glob("/usr/local/cuda*/nvvm/libdevice/libdevice*.bc"))
print(cands)

src = cands[0]
os.makedirs("/usr/local/cuda-11.0/nvvm/libdevice", exist_ok=True)
dst = "/usr/local/cuda-11.0/nvvm/libdevice/libdevice.10.bc"
if not os.path.exists(dst):
    os.symlink(src, dst)
print(os.path.exists(dst))

['/usr/local/envs/sam/lib/libdevice.10.bc']
True


In [8]:
!/usr/local/envs/sam/bin/python -c "import jax; print(jax.random.normal(jax.random.PRNGKey(0), (3,)))"

[ 1.8160863  -0.48262316  0.33988908]


In [9]:
!mkdir -p /content/runs
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 \
  --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sam2 \
  --num_epochs 2 \
  --batch_size 256 \
  --learning_rate 0.1 \
  --weight_decay 0.0005 \
  --sam_rho 0.05 \
  --run_seed 0 2>&1 | tee /content/runs/smoke_sam2.log

2026-10-20 20:55:23.294101: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/lib:/usr/lib64-nvidia
2026-10-20 20:55:25.009924: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/lib:/usr/lib64-nvidia
2026-10-20 20:55:25.010045: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcublas.so.10'; dlerror: libcublas.so.10: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/u

In [10]:
!mamba list -n sam | grep -i -E "cudnn|cudatoolkit"
!ls /usr/local/envs/sam/lib | grep -i cudnn
!find / -name "libcudnn.so*" 2>/dev/null | head

  cudatoolkit               11.0.3          h7761cd4_13           conda-forge
  cudnn                     9.10.1.4        haad7af6_0            conda-forge
  libcudnn                  9.10.1.4        h7d33bf5_0            conda-forge
  libcudnn-dev              9.10.1.4        h0fdc2d1_0            conda-forge
libcudnn_adv.so
libcudnn_adv.so.9
libcudnn_adv.so.9.10.1
libcudnn_cnn.so
libcudnn_cnn.so.9
libcudnn_cnn.so.9.10.1
libcudnn_engines_precompiled.so
libcudnn_engines_precompiled.so.9
libcudnn_engines_precompiled.so.9.10.1
libcudnn_engines_runtime_compiled.so
libcudnn_engines_runtime_compiled.so.9
libcudnn_engines_runtime_compiled.so.9.10.1
libcudnn_graph.so
libcudnn_graph.so.9
libcudnn_graph.so.9.10.1
libcudnn_heuristic.so
libcudnn_heuristic.so.9
libcudnn_heuristic.so.9.10.1
libcudnn_ops.so
libcudnn_ops.so.9
libcudnn_ops.so.9.10.1
libcudnn.so
libcudnn.so.9
libcudnn.so.9.10.1
/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn.so.9
/usr/local/pkgs/libcudnn-dev-9.10.1.4-

In [11]:
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/sam/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

!/usr/local/envs/sam/bin/python -c "import jax.numpy as jnp; from jax import lax; x=jnp.ones((1,8,8,3)); k=jnp.ones((3,3,3,4)); print(lax.conv_general_dilated(x,k,(1,1),'SAME',dimension_numbers=('NHWC','HWIO','NHWC')).shape)" 2>&1 | tail -5

  File "/usr/local/envs/sam/lib/python3.8/site-packages/jax/interpreters/xla.py", line 236, in apply_primitive
    return compiled_fun(*args)
  File "/usr/local/envs/sam/lib/python3.8/site-packages/jax/interpreters/xla.py", line 351, in _execute_compiled_primitive
    out_bufs = compiled.execute(input_bufs)
RuntimeError: Unimplemented: DNN library is not found.


In [13]:
!mamba install -n sam -c conda-forge "cudnn=8.0.5" -y
!ls /usr/local/envs/sam/lib | grep "libcudnn.so"

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
[+] 0.0s

Pinned packages:

  - python=3.8


Transaction

  Prefix: /usr/local/envs/sam

  All requested packages already installed


Transaction starting
[+] 0.0s

Transaction finished

libcudnn.so
libcudnn.so.8
libcudnn.so.8.0.5
libcudnn.so.9
libcudnn.so.9.10.1


In [14]:
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/sam/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

!/usr/local/envs/sam/bin/python -c "import jax.numpy as jnp; from jax import lax; x=jnp.ones((1,8,8,3)); k=jnp.ones((3,3,3,4)); print(lax.conv_general_dilated(x,k,(1,1),'SAME',dimension_numbers=('NHWC','HWIO','NHWC')).shape)" 2>&1 | tail -5

(1, 8, 8, 4)


In [15]:
%cd /content/work
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sam3 \
  --num_epochs 2 --batch_size 256 \
  --learning_rate 0.1 --weight_decay 0.0005 \
  --sam_rho 0.05 --run_seed 0 > /content/runs/smoke_sam3.log 2>&1
!tail -30 /content/runs/smoke_sam3.log
!grep -h "finished in" /content/runs/smoke_sam3.log

/content/work
2026-10-20 21:06:20.975827: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/lib:/usr/lib64-nvidia
2026-10-20 21:06:22.934597: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcudart.so.10.1'; dlerror: libcudart.so.10.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/envs/sam/lib:/usr/local/lib:/usr/lib64-nvidia
2026-10-20 21:06:22.934722: W tensorflow/stream_executor/platform/default/dso_loader.cc:59] Could not load dynamic library 'libcublas.so.10'; dlerror: libcublas.so.10: cannot open shar

In [16]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/sam_runs /content/drive/MyDrive/tfds_cache
!cp -r /content/runs/* /content/drive/MyDrive/sam_runs/
!cp -r /root/tensorflow_datasets/* /content/drive/MyDrive/tfds_cache/
!ls /content/drive/MyDrive/sam_runs /content/drive/MyDrive/tfds_cache

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/sam_runs:
smoke_sam   smoke_sam2.log  smoke_sam3.log  smoke_sgd.log
smoke_sam2  smoke_sam3	    smoke_sam.log

/content/drive/MyDrive/tfds_cache:
cifar10  downloads


In [17]:
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/envs/sam/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [18]:
%cd /content/work
!/usr/local/envs/sam/bin/python -m sam.sam_jax.train \
  --dataset cifar10 --model_name WideResnet28x10 \
  --output_dir /content/runs/smoke_sgd3 \
  --num_epochs 1 --batch_size 256 \
  --learning_rate 0.1 --weight_decay 0.0005 \
  --sam_rho 0.0 --run_seed 0 > /content/runs/smoke_sgd3.log 2>&1
!grep -h "finished in" /content/runs/smoke_sgd3.log

/content/work
I1020 21:31:48.645497 137853772924736 flax_training.py:858] Epoch 0 finished in 257.41s.


In [19]:
!cp -r /content/runs/* /content/drive/MyDrive/sam_runs/
!ls /content/drive/MyDrive/sam_runs

smoke_sam   smoke_sam2.log  smoke_sam3.log  smoke_sgd3	    smoke_sgd.log
smoke_sam2  smoke_sam3	    smoke_sam.log   smoke_sgd3.log


In [21]:
%%writefile /content/read_tb.py
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from tensorboard.util import tensor_util
import os

runs = {
    "SAM": "/content/runs/smoke_sam3/lr_0.1/wd_0.0005/rho_0.05/seed_0",
    "SGD": "/content/runs/smoke_sgd3/lr_0.1/wd_0.0005/rho_0.0/seed_0",
}
for name, d in runs.items():
    print("==", name, "exists:", os.path.isdir(d))
    ea = EventAccumulator(d, size_guidance={"tensors": 0, "scalars": 0})
    ea.Reload()
    tags = ea.Tags()
    print("scalars:", tags["scalars"])
    print("tensors:", tags["tensors"])
    for t in tags["tensors"]:
        ev = ea.Tensors(t)
        vals = [(e.step, round(float(tensor_util.make_ndarray(e.tensor_proto)), 4)) for e in ev]
        print(t, vals[-3:])
    for t in tags["scalars"]:
        ev = ea.Scalars(t)
        print(t, [(e.step, round(e.value, 4)) for e in ev[-3:]])

Writing /content/read_tb.py


In [22]:
!/usr/local/envs/sam/bin/python /content/read_tb.py 2>&1 | grep -v -E "dso_loader|cudart|Skipping"

== SAM exists: True
scalars: []
tensors: ['gradient_norm', 'param_norm', 'train_error_rate', 'train_loss', 'learning_rate', 'test_error_rate', 'test_loss']
gradient_norm [(195, 1.5601), (390, 1.3698)]
param_norm [(195, 155.5004), (390, 148.2844)]
train_error_rate [(195, 0.6183), (390, 0.4269)]
train_loss [(195, 1.8523), (390, 1.1979)]
learning_rate [(195, 0.0504), (390, 0.0)]
test_error_rate [(195, 0.46), (390, 0.3511)]
test_loss [(195, 1.2891), (390, 0.9954)]
== SGD exists: True
scalars: []
tensors: ['gradient_norm', 'param_norm', 'train_error_rate', 'train_loss', 'learning_rate', 'test_error_rate', 'test_loss']
gradient_norm [(195, 1.6024)]
param_norm [(195, 156.928)]
train_error_rate [(195, 0.5631)]
train_loss [(195, 1.6669)]
learning_rate [(195, 0.0)]
test_error_rate [(195, 0.4352)]
test_loss [(195, 1.2124)]


In [23]:
!grep -n -E "PRNGKey|seed|shuffle" /content/work/sam/sam_jax/models/load_model.py /content/work/sam/sam_jax/datasets/dataset_source.py
!grep -n -E "BatchNorm|axis_name|batch_norm" /content/work/sam/sam_jax/models/wide_resnet.py

/content/work/sam/sam_jax/models/load_model.py:56:    with flax.nn.stochastic(jax.random.PRNGKey(0)):
/content/work/sam/sam_jax/models/load_model.py:116:    prng_key = random.PRNGKey(0)
/content/work/sam/sam_jax/datasets/dataset_source.py:116:    ds = self._train_ds.shuffle(50000)


In [24]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!/usr/local/envs/sam/bin/python -m pip list 2>/dev/null | grep -i -E "^(jax|jaxlib|flax|numpy|tensorflow|absl-py) "
!mamba list -n sam | grep -i -E "cudatoolkit|^cudnn |libcudnn "

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07
absl-py                  0.10.0
flax                     0.2.2
jax                      0.2.6
jaxlib                   0.1.57+cuda110
numpy                    1.18.5
tensorflow               2.3.1
  cudatoolkit               11.0.3          h7761cd4_13           conda-forge
  libcudnn                  9.10.1.4        h7d33bf5_0            conda-forge


In [25]:
!grep -rn -E "BatchNorm|batch_norm|axis_name" /content/work/sam/sam_jax/models/

grep: /content/work/sam/sam_jax/models/__pycache__/utils.cpython-38.pyc: binary file matches
/content/work/sam/sam_jax/models/utils.py:36:  """Applies BatchNorm and then (optionally) ReLU.
/content/work/sam/sam_jax/models/utils.py:43:    name: How to name the BatchNorm layer.
/content/work/sam/sam_jax/models/utils.py:46:    The input tensor where BatchNorm and (optionally) ReLU where applied.
/content/work/sam/sam_jax/models/utils.py:48:  batch_norm = nn.BatchNorm.partial(
/content/work/sam/sam_jax/models/utils.py:52:  x = batch_norm(x, name=name)


In [26]:
!sed -n 36,56p /content/work/sam/sam_jax/models/utils.py

  """Applies BatchNorm and then (optionally) ReLU.

  Args:
    x: Tensor on which the activation should be applied.
    train: If False, will use the moving average for batch norm statistics.
        Else, will use statistics computed on the batch.
    apply_relu: Whether or not ReLU should be applied after batch normalization.
    name: How to name the BatchNorm layer.

  Returns:
    The input tensor where BatchNorm and (optionally) ReLU where applied.
  """
  batch_norm = nn.BatchNorm.partial(
      use_running_average=not train,
      momentum=_BATCHNORM_MOMENTUM,
      epsilon=_BATCHNORM_EPSILON)
  x = batch_norm(x, name=name)
  if apply_relu:
    x = jax.nn.relu(x)
  return x

